In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.utils import image_dataset_from_directory
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from ultralytics import YOLO
import glob
from PIL import Image
import IPython.display as display
import cv2
import seaborn as sns

### Import and Check the GPU

In [ ]:
%pip install pandas
%pip install matplotlib
%pip install seaborn
%pip install scikit-learn

In [ ]:
gpus = tf.config.list_physical_devices('GPU')

In [ ]:
gpus

### Define, Import and Check dataset

In [ ]:
# Define image dimensions and batch size
target_height = 101 # Target height for resizing
target_width = 101  # Target width for resizing
batch_size = 32
num_classes = 2
seed = 42  # For reproducibility

In [ ]:
# Define your directories
train_dir = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\train data"
validation_dir = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\validation data"
test_dir = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\test data"

# Check if directories exist
print(f"Train directory exists: {os.path.exists(train_dir)}")
print(f"Validation directory exists: {os.path.exists(validation_dir)}")
print(f"Test directory exists: {os.path.exists(test_dir)}")

# Check what's inside each directory
def check_directory_contents(directory_path, dir_name):
    print(f"\n--- {dir_name} Directory Contents ---")
    if os.path.exists(directory_path):
        items = os.listdir(directory_path)
        print(f"Number of items: {len(items)}")
        for item in items:
            item_path = os.path.join(directory_path, item)
            if os.path.isdir(item_path):
                # It's a subdirectory (should be class names)
                image_files = [f for f in os.listdir(item_path) 
                             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
                print(f"📁 Class folder: {item} - {len(image_files)} images")
            else:
                # It's a file
                print(f"📄 File: {item}")
    else:
        print("Directory does not exist!")

check_directory_contents(train_dir, "Train")
check_directory_contents(validation_dir, "Validation")
check_directory_contents(test_dir, "Test")

In [ ]:
datagen_train = image_dataset_from_directory(train_dir,
                                            color_mode='grayscale',
                                            image_size=(target_height,target_width),
                                            batch_size=None,
                                            label_mode='categorical'
                                            )

datagen_val = image_dataset_from_directory(validation_dir,
                                           color_mode='grayscale',
                                           image_size=(target_height,target_width),
                                           batch_size=None,
                                           label_mode='categorical'
                                           )
datagen_test = image_dataset_from_directory(test_dir,
                                            color_mode='grayscale',
                                            image_size=(target_height,target_width),
                                            batch_size=None,
                                            label_mode='categorical'
                                            )

In [ ]:
# Check the class names and number of classes

df = pd.DataFrame({"train": datagen_train.class_names,
                   "validation": datagen_val.class_names,
                   "test": datagen_test.class_names})
df

In [ ]:
print(f"Number of training samples: {len(datagen_train)}")
print(f"Number of validation samples: {len(datagen_val)}")
print(f"Number of test samples: {len(datagen_test)}")

In [ ]:
print("Training set")
print(f"Class names: {datagen_train.class_names}")
num_classes = len(datagen_train.class_names)
print(f"Number of classes: {len(datagen_train.class_names)}")

In [ ]:
# Recreate datasets with proper categorical encoding
datagen_train = image_dataset_from_directory(
    train_dir,
    color_mode='grayscale',
    image_size=(target_height, target_width),
    batch_size=32,  # Make sure you have a batch size
    label_mode='categorical',  # This should create one-hot encoding
    class_names=['water', 'dont water'],  # Explicitly define 2 classes
    shuffle=True
)

In [ ]:
for images, labels in datagen_train.take(1):
    print(f"Image shape: {images.shape}")
    print(f"Label shape: {labels.shape}")
    break

In [ ]:
# Recreate datasets with proper categorical encoding
datagen_test = image_dataset_from_directory(
    test_dir,
    color_mode='grayscale',
    image_size=(target_height, target_width),
    batch_size=32,  # Make sure you have a batch size
    label_mode='categorical',  # This should create one-hot encoding
    class_names=['water', 'dont water'],  # Explicitly define 2 classes
    shuffle=True
)

In [ ]:
for images, labels in datagen_train.take(1):
    print(f"Image shape: {images.shape}")
    print(f"Label shape: {labels.shape}")
    break

In [ ]:
# Recreate datasets with proper categorical encoding
datagen_val = image_dataset_from_directory(
    validation_dir,
    color_mode='grayscale',
    image_size=(target_height, target_width),
    batch_size=32,  # Make sure y_drou have a batch size
    label_mode='categorical',  # This should create one-hot encoding
    class_names=['water', 'dont water'],  # Explicitly define 2 classes
    shuffle=True
)

In [ ]:
for images, labels in datagen_train.take(1):
    print(f"Image shape: {images.shape}")
    print(f"Label shape: {labels.shape}")
    break

In [ ]:
class_names = datagen_train.class_names
class_counts = [0] * len(class_names)

for images, labels in datagen_train:
    class_counts[np.argmax(labels.numpy())] += 1

class_distribution = pd.DataFrame({"Class": class_names, "Count": class_counts})

In [ ]:
class_names_val = datagen_val.class_names
class_counts_val = [0] * len(class_names_val)

for images, labels in datagen_val:
    class_counts_val[np.argmax(labels.numpy())] += 1

class_distribution_val = pd.DataFrame({"Class": class_names_val, "Count": class_counts_val})

In [ ]:
class_names_test = datagen_test.class_names
class_counts_test = [0] * len(class_names_test)

for images, labels in datagen_test:
    class_counts_test[np.argmax(labels.numpy())] += 1

class_distribution_test = pd.DataFrame({"Class": class_names_test, "Count": class_counts_test})

-- Visualisation of the train dataset --

In [ ]:
ClassNames = datagen_train.class_names
print(ClassNames)

In [ ]:
plt.figure(figsize=(10, 10))
for images, labels in datagen_train.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"), cmap='gray')
        plt.title(ClassNames[np.argmax(labels[i].numpy())])
        plt.axis("off")

-- Adding batch size -- 

batch_size controls how many images are loaded and processed together at once when training or evaluating the model

> When training a neutral network, dataset is usually too large to fit into memory all at once, so instead of laoding all the images at once, the data is divided into batches

In [ ]:
datagen_train = image_dataset_from_directory(train_dir,
                                            color_mode='grayscale',
                                            image_size=(target_height,target_width),
                                            batch_size=10000,
                                            label_mode='categorical'
                                            )

datagen_val = image_dataset_from_directory(validation_dir,
                                           color_mode='grayscale',
                                           image_size=(target_height,target_width),
                                           batch_size=10000,
                                           label_mode='categorical'
                                           )

datagen_test = image_dataset_from_directory(test_dir,
                                            color_mode='grayscale',
                                            image_size=(target_height,target_width),
                                            batch_size=10000,
                                            label_mode='categorical'
                                            )

In [ ]:
for X, y in datagen_train:
    X_train = X / 255
    y_train = y
print(X_train.shape)
print(y_train.shape)

for X, y in datagen_val:
    X_val = X / 255
    y_val = y
print(X_val.shape)
print(y_val.shape)

for X, y in datagen_test:
    X_test = X / 255
    y_test = y
print(X_test.shape)
print(y_test.shape)

### Training Classification Model 

In [ ]:
model = Sequential()
model.add(layers.Flatten(input_shape=(target_height, target_width, 1)))  # Flatten the input
model.add(layers.Dense(units=800, activation="relu"))
model.add(layers.Dropout(0.5))  # Dropout layer to prevent overfitting
model.add(layers.Dense(units=64, activation="relu"))
model.add(layers.Dropout(0.2))  # Dropout layer to prevent overfitting
model.add(layers.Dense(units=num_classes, activation="softmax"))
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=128, validation_data=(X_val, y_val))

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

-- Saving model results --

In [ ]:
model.save('Mushroom_classification_model.h5')

In [ ]:
loaded_model = keras.models.load_model('Mushroom_classification_model.h5')

In [ ]:
test_loss, test_accuracy = loaded_model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss}")
print(f"Test Accuracy: {test_accuracy}")

#### Labelling Detection Dataset

In [ ]:
pip install labelImg

In [ ]:
!labelImg

### Creating Detection Model

In [ ]:
pip install ultralytics

In [ ]:
path: r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\detection_dataset"
train: r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\detection_dataset\train\images"
val: r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\detection_dataset\validation\images"
nc: 2 # nc: number of classes
names: ["water", "dont water"]

In [ ]:
yaml_content = """\
path: C:\\Users\\xy07c\\OneDrive\\Desktop\\ST1504 DEEP LEARNING\\TF SCALE\\detection_dataset
train: C:\\Users\\xy07c\\OneDrive\\Desktop\\ST1504 DEEP LEARNING\\TF SCALE\\detection_dataset\\train\\images
val: C:\\Users\\xy07c\\OneDrive\\Desktop\\ST1504 DEEP LEARNING\\TF SCALE\\detection_dataset\\validation\\images
nc: 2
names: ["water", "dont water"]"""

In [ ]:
yaml_path = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\mushroom.yaml"

In [ ]:
with open(yaml_path, "w") as f:
    f.write(yaml_content)

-- epochs 50 --

In [ ]:
!yolo detect train data="C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\mushroom.yaml" model=yolov8n.pt epochs=50 imgsz=640

### Evaluation using YOLO's Built-In Evaluation 

In [ ]:
!yolo detect val model=runs/detect/train30/weights/best.pt data=mushroom.yaml

In [ ]:
model = YOLO("runs/detect/train30/weights/best.pt")

In [ ]:
metrics = model.val()

In [ ]:
precision = np.mean(metrics.box.p)    
recall = np.mean(metrics.box.r)         
map50 = np.mean(metrics.box.map50)      
map5095 = np.mean(metrics.box.map) 

In [ ]:
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"mAP50 (Accuracy equivalent): {map50:.3f}")
print(f"mAP50-95: {map5095:.3f}")

In [ ]:
img_path = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\runs\detect\val31"

In [ ]:
for img_path in glob.glob(img_path + "/*.jpg"):
    print(f"Showing: {img_path}")
    display.display(Image.open(img_path))

In [ ]:
results = model.predict(source=r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\detection_dataset\test\images", save=False)

In [ ]:
colors = {
    "water": (139, 0, 0),       # Dark Blue
    "dont water": (0, 0, 255)   # Red
}

# Creating a new folder for the new images (images with the new coloured labels)
os.makedirs("detection_results", exist_ok=True)

# Loop through each prediction
for i, r in enumerate(results):
    img = r.orig_img.copy()
    for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
        label = model.names[int(cls)]
        color = colors.get(label, (255, 255, 255))  # white default

        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        text = f"{label} {conf:.1f}"
        cv2.putText(img, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, color, 2)

    save_path = f"detection_results/results_{i}.jpg"
    cv2.imwrite(save_path, img)
    print(f"Saved image to: {save_path}")

In [ ]:
colors = {
    "water": (139, 0, 0),       # Dark Blue
    "dont water": (0, 0, 255)   # Red
}

os.makedirs("detection_results", exist_ok=True)

BOX_THICKNESS = 20       
FONT_SCALE = 5        
FONT_THICKNESS = 10         
FONT_STYLE = cv2.FONT_HERSHEY_DUPLEX  

for i, r in enumerate(results):
    img = r.orig_img.copy()
    for box, cls, conf in zip(r.boxes.xyxy, r.boxes.cls, r.boxes.conf):
        label = model.names[int(cls)]
        color = colors.get(label, (255, 255, 255))  # white default

        x1, y1, x2, y2 = map(int, box)
        
        cv2.rectangle(img, (x1, y1), (x2, y2), color, BOX_THICKNESS)
        
        text = f"{label} {conf:.1f}"
        
        text_size = cv2.getTextSize(text, FONT_STYLE, FONT_SCALE, FONT_THICKNESS)[0]
        
        cv2.rectangle(img, 
                     (x1, y1 - text_size[1] - 10), 
                     (x1 + text_size[0], y1), 
                     color, -1)
        
        cv2.putText(img, text, (x1, y1 - 5), FONT_STYLE,
                    FONT_SCALE, (255, 255, 255), FONT_THICKNESS)  # White text

    save_path = f"detection_results/results_{i}.jpg"
    cv2.imwrite(save_path, img)
    print(f"Saved image to: {save_path}")

In [ ]:
# Auto-tag saved images with growth stage based on YOLO detections
# water label -> mature stage | dont water label -> small_medium | no detections -> none
import os

LABEL_TO_STAGE = {"water": "mature", "dont water": "small_medium"}

for i, r in enumerate(results):
    old_path = f"detection_results/results_{i}.jpg"
    if not os.path.exists(old_path):
        continue

    # Pick the most frequent label in this image
    labels = [model.names[int(cls)] for cls in r.boxes.cls]
    if not labels:
        stage = "none"
    else:
        stage = LABEL_TO_STAGE.get(max(set(labels), key=labels.count), "unknown")

    new_path = f"detection_results/{stage}_results_{i}.jpg"
    os.rename(old_path, new_path)
    print(f"Renamed: {old_path} -> {new_path}")


In [ ]:
img_path = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\detection_results"

In [ ]:
for img_path in glob.glob(img_path + "/*.jpg"):
    print(f"Showing: {img_path}")
    display.display(Image.open(img_path))

In [ ]:
confusion_matrix_path = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\runs\detect\train30\confusion_matrix.png"
confusion_matrix_img = plt.imread(confusion_matrix_path)

plt.figure(figsize=(10, 8))
plt.imshow(confusion_matrix_img)
plt.axis('off')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
confusion_matrix_path = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\runs\detect\train30\confusion_matrix.png"
confusion_matrix_img = plt.imread(confusion_matrix_path)

gray = np.mean(confusion_matrix_img[:, :, :3], axis=2)

plt.figure(figsize=(10, 8))
plt.imshow(gray, cmap='Reds')
plt.axis('off')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
result_path = r"C:\Users\xy07c\OneDrive\Desktop\ST1504 DEEP LEARNING\TF SCALE\runs\detect\train30\results.png"
result_img = plt.imread(result_path)
plt.figure(figsize=(10, 8))
plt.imshow(result_img)
plt.axis('off')
plt.title('Results')
plt.show()

In [ ]:
!jupyter nbconvert --to html  Classification_Detection_Model.ipynb

### Send Detection Results to FastAPI Frontend

In [ ]:
import requests, os, glob
from IPython.display import Image, display

# URL of the running FastAPI backend
API_URL = "http://localhost:8000/api/detect"

def send_image_to_api(image_path):
    with open(image_path, "rb") as f:
        resp = requests.post(API_URL, files={"file": (os.path.basename(image_path), f, "image/jpeg")})
    resp.raise_for_status()
    return resp.json()

# Send all saved detection result images to the frontend API
image_paths = glob.glob("detection_results/*.jpg") + glob.glob("detection_results/*.png")

if not image_paths:
    print("No images found in detection_results/ — run the detection cells above first.")
else:
    for path in image_paths:
        print(f"
Sending: {path}")
        result = send_image_to_api(path)
        detections = result.get("detections", [])
        print(f"  {len(detections)} detection(s):")
        for d in detections:
            print(f"    [{d['label']}] confidence={d['confidence']}  box={d['box']}")
        display(Image(filename=path, width=400))


### (Optional) Start FastAPI Backend from Notebook
> Run this cell only if the backend is not already running separately.
> Update  to where you cloned the IIICe-SP repo.

In [ ]:
import subprocess, time, os

repo_path = r"C:\path	o\IIICe-SP"  # <-- update this
weights_path = os.path.join(os.getcwd(), "runs", "detect", "train30", "weights", "best.pt")

env = os.environ.copy()
env["SPOTSHROOMS_YOLO_WEIGHTS"] = weights_path

proc = subprocess.Popen(
    ["uvicorn", "api.main:app", "--reload", "--port", "8000"],
    cwd=repo_path,
    env=env
)
time.sleep(3)
print(f"API running at http://localhost:8000")
print(f"Using YOLO weights: {weights_path}")
